# SAE Final Prompt Construction For One Pair



In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import pandas as pd
import torch
import yaml


PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "SAE").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from SAE.SAE_training.model import SAEConfig
from SAE.LLM_pipeline.utils.final_interpretation import (
    build_final_interpretation_prompt,
    format_dictionary_atom_table,
    format_feature_evidence_blocks,
)
from SAE.LLM_pipeline.utils.general import sanitize_groundtruth_text
from SAE.LLM_pipeline.utils.llm_strategy import final_strategy_text, run_llm_strategy, strategy_call_count, strategy_report_text, strategy_total_tokens
from SAE.LLM_pipeline.utils.nli import compare_interpretation_to_groundtruth, nli_model_path
from prompt_api.client import AigcBestChatClient

print("Project root:", PROJECT_ROOT)

## Load SAE Artifacts

The final prompt uses only compact SAE dictionary atom summaries from the explainer run. Ground truth is loaded only after generation for comparison.


In [ ]:
RUN_LLM = False

NOTEBOOK_CONFIG = PROJECT_ROOT / "src" / "SAE" / "LLM_pipeline" / "config" / "explainer_simulator_config.yaml"
notebook_cfg = yaml.safe_load(NOTEBOOK_CONFIG.read_text(encoding="utf-8"))
train_cfg = yaml.safe_load((PROJECT_ROOT / notebook_cfg["train_config"]).read_text(encoding="utf-8"))
prompt_api_config = PROJECT_ROOT / notebook_cfg["prompt_api_config"]
model_cfg = SAEConfig(**dict(train_cfg["model"]))
paths_cfg = train_cfg["paths"]
scope_cfg = train_cfg["scope"]

target_cfg = notebook_cfg["target"]
TARGET_PRIMARY = str(target_cfg["primary_gene"])
TARGET_PARTNER = str(target_cfg["partner_gene"])
TARGET_CANCER = str(target_cfg["cancer"])
EXISTS_GROUNDTRUTH = bool(target_cfg["exists_groundtruth"])

strategy_cfg = notebook_cfg["llm_strategy"]
FINAL_INTERPRETATION_STRATEGY = str(strategy_cfg["final_interpretation"])
TRAIN_SCOPE = str(scope_cfg["cancer"])
run_name = f"hidden{model_cfg.d_hidden}_gate{model_cfg.gate_weight}_orth{model_cfg.orth_weight}_k{model_cfg.topk}"
output_base_dir = Path(paths_cfg["output_base_dir"]).expanduser().resolve()
slformer_output_subdir = str(paths_cfg["slformer_output_subdir"])
output_dir = output_base_dir / slformer_output_subdir / TRAIN_SCOPE / run_name / "explanations" / f"{TARGET_CANCER}_{TARGET_PRIMARY}-{TARGET_PARTNER}"
final_dir = output_dir / "final_prompt"
final_dir.mkdir(parents=True, exist_ok=True)

state = json.loads((output_dir / "interpretation_state.json").read_text(encoding="utf-8"))
feature_rank = pd.read_csv(output_dir / "feature_rank.csv")
summary = pd.read_csv(output_dir / "llm_interpretation_summary.csv", keep_default_na=False)
summary["confidence"] = summary["confidence"].astype(str).str.strip().str.lower()

if EXISTS_GROUNDTRUTH:
    groundtruth_text = sanitize_groundtruth_text((
        (PROJECT_ROOT / "data" / "explanation_groundtruth" / f"{TARGET_PRIMARY}-{TARGET_PARTNER}" / "explanation.txt")
        .read_text(encoding="utf-8")
    ))
else:
    groundtruth_text = ""
    print("exists_groundtruth is False; ground-truth text not loaded; local NLI will be skipped.")

template = (PROJECT_ROOT / "src" / "SAE" / "doc" / "prompts" / "final_interpretation_prompt_template.txt").read_text(encoding="utf-8")

print("Loaded features:", len(summary))
print("RUN_LLM:", RUN_LLM)
display(summary)


## Construct Final Prompt

The prompt table contains one row per selected SAE dictionary atom: atom ID, biological label, local geometry, and exemplar-derived rationale.


In [ ]:
dictionary_atom_table = format_dictionary_atom_table(feature_rank, summary.to_dict(orient="records"))
feature_evidence = format_feature_evidence_blocks(summary.to_dict(orient="records"))

final_prompt = build_final_interpretation_prompt(
    template=template,
    target_primary=TARGET_PRIMARY,
    target_partner=TARGET_PARTNER,
    cancer=TARGET_CANCER,
    target_score=float(state["target"]["score"]),
    dictionary_atom_table=dictionary_atom_table,
    feature_evidence=feature_evidence,
)
if RUN_LLM:
    (final_dir / "final_interpretation_prompt.md").write_text(final_prompt, encoding="utf-8")
print("Prompt characters:", len(final_prompt))
print(final_prompt[:2000])


## Generate Final Interpretation

This cell runs the configured final-interpretation LLM strategy. When `final_interpretation: cove`, the strategy runs the baseline draft, self-refine, and CoVe verification. Intermediate waits between self-refine and CoVe API calls are read from `src/prompt_api/client_config.yaml`; the final response file includes the generated answer followed by the trace.


In [ ]:
final_md = final_dir / "final_interpretation.md"
final_response = final_dir / "final_interpretation_response.md"
final_prompt_path = final_dir / "final_interpretation_prompt.md"

if not RUN_LLM:
    final_text = final_md.read_text(encoding="utf-8")
    final_report = final_response.read_text(encoding="utf-8") if final_response.exists() else final_text
    print("RUN_LLM=False; loaded completed final interpretation without API calls.")
else:
    client = AigcBestChatClient(config_path=prompt_api_config)
    max_call_s = int(client.settings.request_retries) * float(client.settings.request_timeout_s) + (int(client.settings.request_retries) - 1) * float(client.settings.retry_wait_s)
    final_call_count = strategy_call_count(
        FINAL_INTERPRETATION_STRATEGY,
        self_refine_rounds=client.settings.self_refine_rounds,
        cove_num_questions=client.settings.cove_num_questions,
    )
    print(f"Planned LLM calls: {final_call_count if RERUN_FINAL_INTERPRETATION else 0} final interpretation strategy calls ({FINAL_INTERPRETATION_STRATEGY})")
    print(f"Configured intermediate waits: self_refine={client.settings.self_refine_wait_s:.0f}s, cove={client.settings.cove_wait_s:.0f}s")

    if not RERUN_FINAL_INTERPRETATION and final_md.exists() and final_response.exists() and final_prompt_path.exists():
        final_text = final_md.read_text(encoding="utf-8")
        final_report = final_response.read_text(encoding="utf-8")
        print("Reused completed final interpretation.")
    else:
        started = time.monotonic()
        final_trace = run_llm_strategy(client, final_prompt, FINAL_INTERPRETATION_STRATEGY)
        final_text = final_strategy_text(final_trace)
        final_report = strategy_report_text(final_trace)
        print(f"Final interpretation done in {time.monotonic() - started:.1f}s, tokens={strategy_total_tokens(final_trace)}")
        final_response.write_text(final_report, encoding="utf-8")
        final_md.write_text(final_text.strip() + "\n", encoding="utf-8")
print(final_report[:4000])


## Compare With Ground Truth Using Local NLI

A local NLI model from `/data/guoyu/HF-models` replaces the previous LLM comparison: curated text entails SAE claims when the explanation is supported, and SAE text entails curated sentences when the final interpretation covers ground-truth mechanisms.

In [ ]:
if not EXISTS_GROUNDTRUTH:
    print("Local NLI disabled because exists_groundtruth is False.")
else:
    NLI_MODEL_ROOT = Path("/data/guoyu/HF-models")
    nli_path = nli_model_path(NLI_MODEL_ROOT)
    nli_device = "cuda" if torch.cuda.is_available() else "cpu"
    print("NLI model:", nli_path)
    print("NLI device:", nli_device)

    nli_table = compare_interpretation_to_groundtruth(
        final_text=final_text,
        groundtruth_text=groundtruth_text,
        model_path=nli_path,
        device=nli_device,
        batch_size=8,
        max_length=2048,
    )
    nli_counts = (
        nli_table.groupby(["comparison", "prediction"])
        .size()
        .rename("n_sentences")
        .reset_index()
        .sort_values(["comparison", "prediction"])
    )

    nli_table_path = final_dir / "nli_groundtruth_comparison.csv"
    nli_table.to_csv(nli_table_path, index=False)

    print("Saved NLI table:", nli_table_path)
    display(nli_counts)
    display(nli_table[["comparison", "prediction", "entailment", "neutral", "contradiction", "hypothesis"]])


## Save Output State

The final output state records paths, generated text, and local NLI comparison outputs for downstream reading.

In [ ]:
print("Saved final prompt:", final_dir / "final_interpretation_prompt.md")
print("Saved final response report:", final_dir / "final_interpretation_response.md")
print("Saved final interpretation markdown:", final_dir / "final_interpretation.md")
